# Task 1.2 — Chunking Strategy Implementation

This notebook implements two chunking strategies:
1. **Recursive Character Text Splitting** (required)
2. **Section-Aware Chunking** (chosen additional strategy)

Preprocessing handles image placeholders and LaTeX formula protection before chunking.

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q --upgrade opentelemetry-api opentelemetry-sdk
!pip install -q chromadb sentence-transformers tiktoken langchain langchain-text-splitters
print("✅ All dependencies installed")

## Cell 2 — Mount Drive + Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_URL  = "https://github.com/RyanChenJung/Rag-Pipeline-Over-AI-Research-Paper.git"
REPO_NAME = "Rag-Pipeline-Over-AI-Research-Paper"

if not os.path.exists(f"/content/{REPO_NAME}"):
    !git clone {REPO_URL}

%cd /content/{REPO_NAME}

!git config --global user.email "your_email@uchicago.edu"
!git config --global user.name "SolaShinData"
!git checkout sola/chunking-embedding

print("✅ Drive mounted and repo ready")

## Cell 3 — Unzip Parsed Data

In [ ]:
import zipfile
import os

# Auto-detect zip file in Assignment3 folder
base     = "/content/drive/MyDrive/[3Q]Capstone1/Assignment3"
zip_path = None
for f in os.listdir(base):
    if f.endswith(".zip"):
        zip_path = os.path.join(base, f)
        print(f"✅ zip found: {zip_path}")

EXTRACT_PATH = "/content/parsed_output"

if zip_path and not os.path.exists(EXTRACT_PATH):
    print("📦 Unzipping...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)
    print("✅ Done!")
else:
    print("✅ Already extracted, skipping")

print("\n📁 Contents:")
for item in sorted(os.listdir(EXTRACT_PATH)):
    print(f"  {item}")

## Cell 4 — Define Preprocessing Functions

Handles two special cases from Marker/MinerU output:
- **Image placeholders**: replaced with semantic `[FIGURE: caption]` text
- **LaTeX formulas**: protected as atomic units to prevent splitting mid-formula

In [ ]:
import re

def preprocess_markdown(text: str) -> str:
    """Clean parsed markdown before chunking.
    - Replace image placeholders with semantic text
    - Protect LaTeX blocks so they never get split across chunks
    """
    # Replace image placeholders: ![caption](path) → [FIGURE: caption]
    text = re.sub(
        r'!\[([^\]]*)\]\([^\)]+\)',
        lambda m: f"[FIGURE: {m.group(1)}]" if m.group(1).strip() else "[FIGURE]",
        text
    )
    # Protect block LaTeX: $$...$$ → keep as one unit
    def protect_block_latex(match):
        inner = match.group(1).replace('\n', ' ')
        return f"$$LATEX_BLOCK_START {inner} LATEX_BLOCK_END$$"
    text = re.sub(r'\$\$(.*?)\$\$', protect_block_latex, text, flags=re.DOTALL)
    # Protect inline LaTeX: $...$ → [LATEX_INLINE: ...]
    text = re.sub(r'\$([^\$\n]+)\$', r'[LATEX_INLINE: \1]', text)
    return text


def restore_latex(text: str) -> str:
    """Restore protected LaTeX markers back to proper format after chunking."""
    text = re.sub(r'\$\$LATEX_BLOCK_START (.*?) LATEX_BLOCK_END\$\$', r'$$\1$$', text)
    text = re.sub(r'\[LATEX_INLINE: (.*?)\]', r'$\1$', text)
    return text

print("✅ Preprocessing functions defined")

## Cell 5 — Load All Documents

In [ ]:
import glob

def load_documents(extract_path: str) -> list:
    """Load all markdown files and return list of dicts with text + metadata."""
    md_files  = glob.glob(f"{extract_path}/**/*.md", recursive=True)
    documents = []
    for path in sorted(md_files):
        with open(path, "r", encoding="utf-8") as f:
            raw_text = f.read()
        cleaned  = preprocess_markdown(raw_text)
        arxiv_id = os.path.basename(path).replace(".md", "").replace(".mmd", "")
        documents.append({
            "arxiv_id"   : arxiv_id,
            "file_path"  : path,
            "text"       : cleaned,
            "char_length": len(cleaned)
        })
    print(f"✅ Loaded {len(documents)} documents")
    return documents


docs    = load_documents(EXTRACT_PATH)
lengths = [d["char_length"] for d in docs]

print(f"\n📊 Corpus Statistics:")
print(f"  Total documents  : {len(docs)}")
print(f"  Total chars      : {sum(lengths):,}")
print(f"  Mean doc length  : {sum(lengths)//len(lengths):,} chars")
print(f"  Max doc length   : {max(lengths):,} chars")
print(f"  Min doc length   : {min(lengths):,} chars")

## Cell 6 — Strategy 1: Recursive Character Text Splitting

In [ ]:
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Use cl100k tokenizer (same as OpenAI embeddings) to count tokens
tokenizer = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    return len(tokenizer.encode(text))

# Config: 512 token chunks with 50 token overlap
CHUNK_SIZE    = 512
CHUNK_OVERLAP = 50

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=count_tokens,
    separators=["\n\n", "\n", ".", " ", ""]
)

recursive_chunks = []
for doc in docs:
    for i, chunk in enumerate(recursive_splitter.split_text(doc["text"])):
        recursive_chunks.append({
            "chunk_id"    : f"{doc['arxiv_id']}_rec_{i}",
            "arxiv_id"    : doc["arxiv_id"],
            "strategy"    : "recursive",
            "chunk_index" : i,
            "text"        : restore_latex(chunk),
            "token_count" : count_tokens(chunk)
        })

print(f"✅ Recursive: {len(recursive_chunks):,} chunks")

## Cell 7 — Strategy 2: Section-Aware Chunking

In [ ]:
section_chunks = []
header_pattern = re.compile(r'^(#{1,3} .+)$', re.MULTILINE)

for doc in docs:
    text       = doc["text"]
    splits     = [m.start() for m in header_pattern.finditer(text)]
    boundaries = [0] + splits + [len(text)]
    sections   = [
        text[boundaries[i]:boundaries[i+1]].strip()
        for i in range(len(boundaries)-1)
        if text[boundaries[i]:boundaries[i+1]].strip()
    ]
    idx = 0
    for section in sections:
        tc = count_tokens(section)
        # Section fits within limit → keep as one chunk
        if tc <= CHUNK_SIZE:
            section_chunks.append({
                "chunk_id"    : f"{doc['arxiv_id']}_sec_{idx}",
                "arxiv_id"    : doc["arxiv_id"],
                "strategy"    : "section_aware",
                "chunk_index" : idx,
                "text"        : restore_latex(section),
                "token_count" : tc
            })
            idx += 1
        # Section too long → fall back to recursive splitting
        else:
            for sub in recursive_splitter.split_text(section):
                section_chunks.append({
                    "chunk_id"    : f"{doc['arxiv_id']}_sec_{idx}",
                    "arxiv_id"    : doc["arxiv_id"],
                    "strategy"    : "section_aware",
                    "chunk_index" : idx,
                    "text"        : restore_latex(sub),
                    "token_count" : count_tokens(sub)
                })
                idx += 1

print(f"✅ Section-aware: {len(section_chunks):,} chunks")

## Cell 8 — Filter Tiny Chunks + Statistics Report

In [ ]:
import statistics

# Remove near-empty chunks (likely whitespace or broken image tags)
MIN_TOKENS = 20
recursive_chunks_clean = [{**c, "text": restore_latex(c["text"])} for c in recursive_chunks if c["token_count"] >= MIN_TOKENS]
section_chunks_clean   = [{**c, "text": restore_latex(c["text"])} for c in section_chunks   if c["token_count"] >= MIN_TOKENS]

print(f"Recursive : {len(recursive_chunks):,} → {len(recursive_chunks_clean):,} after filtering")
print(f"Section   : {len(section_chunks):,} → {len(section_chunks_clean):,} after filtering")


def report_stats(chunks: list, strategy_name: str):
    """Print chunk statistics for writeup reporting."""
    token_counts   = [c["token_count"] for c in chunks]
    chunks_per_doc = {}
    for c in chunks:
        chunks_per_doc[c["arxiv_id"]] = chunks_per_doc.get(c["arxiv_id"], 0) + 1
    cpd = list(chunks_per_doc.values())

    print(f"\n{'='*50}")
    print(f"📊 Strategy: {strategy_name}")
    print(f"{'='*50}")
    print(f"  Total chunks          : {len(chunks):,}")
    print(f"  --- Chunk Length (tokens) ---")
    print(f"  Mean                  : {statistics.mean(token_counts):.1f}")
    print(f"  Median                : {statistics.median(token_counts):.1f}")
    print(f"  Std Dev               : {statistics.stdev(token_counts):.1f}")
    print(f"  Min / Max             : {min(token_counts)} / {max(token_counts)}")
    print(f"  --- Chunks per Document ---")
    print(f"  Mean                  : {statistics.mean(cpd):.1f}")
    print(f"  Median                : {statistics.median(cpd):.1f}")
    print(f"  Min / Max             : {min(cpd)} / {max(cpd)}")

report_stats(recursive_chunks_clean, "Recursive Character Splitting")
report_stats(section_chunks_clean,   "Section-Aware Splitting")

## Cell 9 — Save Chunk Files

In [ ]:
import json

OUTPUT_DIR = "/content/Rag-Pipeline-Over-AI-Research-Paper/part1_2_chunking"
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(f"{OUTPUT_DIR}/chunks_recursive_clean.json", "w") as f:
    json.dump(recursive_chunks_clean, f, indent=2)

with open(f"{OUTPUT_DIR}/chunks_section_clean.json", "w") as f:
    json.dump(section_chunks_clean, f, indent=2)

print("✅ Chunk files saved")
print(f"  chunks_recursive_clean.json : {len(recursive_chunks_clean):,} chunks")
print(f"  chunks_section_clean.json   : {len(section_chunks_clean):,} chunks")